## Gradient Accumulation

### What is Gradient Accumulation?
Gradient accumulation allows you to simulate training with a larger batch size than what fits in your GPU memory by accumulating gradients over multiple mini-batches before updating the model weights.

### How It Works:

Memory-limited scenario:<br>
┌─────────────────────────────────────────────────────┐<br>
│ Want: Batch size 8, but only have memory for 1      │<br>
└─────────────────────────────────────────────────────┘<br>

#### Normal Training (without gradient accumulation):
Batch 1 [sample] → gradients → update weights<br>
Batch 2 [sample] → gradients → update weights<br>  
Batch 3 [sample] → gradients → update weights<br>
(Each update uses gradients from only 1 sample)<br>

#### With Gradient Accumulation:
Mini-batch 1 [sample] → gradients → accumulate<br>
Mini-batch 2 [sample] → gradients → accumulate<br>  
Mini-batch 3 [sample] → gradients → accumulate<br>
Mini-batch 4 [sample] → gradients → accumulate<br>
Mini-batch 5 [sample] → gradients → accumulate<br>
Mini-batch 6 [sample] → gradients → accumulate<br>
Mini-batch 7 [sample] → gradients → accumulate<br>
Mini-batch 8 [sample] → gradients → accumulate → UPDATE WEIGHTS<br>
(One update uses gradients from 8 samples combined)<br>

#### Example Breakdown:
```python
# In the training arguments:
per_device_train_batch_size = 1      # Physical batch size (fits in memory)
gradient_accumulation_steps = 8      # Accumulate over 8 mini-batches

# Effective batch size = 1 × 8 = 8
```

These two setups are mathematically equivalent:
```python
# Option A: Large batch (if you had enough memory)
per_device_train_batch_size = 8
gradient_accumulation_steps = 1

# Option B: Small batch with accumulation  
per_device_train_batch_size = 1
gradient_accumulation_steps = 8
```

#### Benefits
- Train with larger effective batch sizes on limited memory
- More stable training
- Better gradient estimates

#### Drawbacks
- Slower training (more forward passes per update)
- Slightly more complex gradient computation
- May use more system RAM for gradient storage

In [ ]:
# Your GPU can only handle batch_size=1 for Llama2 7B
# But you want the training effect of batch_size=8

training_args = TrainingArguments(
    per_device_train_batch_size=1,    # What fits in memory
    gradient_accumulation_steps=8,    # Accumulate 8 mini-batches
    # Effective batch size = 1 × 8 = 8
)

# During training, this happens:
for step in range(training_steps):
    total_loss = 0
    
    # Accumulate gradients over 8 mini-batches
    for i in range(8):
        mini_batch = get_next_batch(size=1)
        loss = model(mini_batch)
        loss = loss / 8  # Scale loss by accumulation steps
        loss.backward()  # Accumulate gradients
        total_loss += loss.item()
    
    # Update weights after accumulating 8 mini-batches
    optimizer.step()
    optimizer.zero_grad()

### Key Takeaway:
Gradient accumulation is like "saving up" gradients from multiple small batches before making one big update to the model. It's essential for training large models on consumer hardware where memory is limited.<br>
Think of it as: "I can't afford to buy 8 items at once, but I can buy 1 item at a time, 8 times, and get the same result!"

## NF4 (4-bit NormalFloat)

NF4 (4-bit NormalFloat) is a quantization technique that significantly reduces memory usage for large language models by representing weights with only 4 bits instead of the typical 16 bits (half-precision) or 32 bits (full-precision).

Here's how NF4 saves memory:

#### Memory Reduction
- Standard FP16: 16 bits per weight
- NF4: 4 bits per weight
- This gives a 4x memory reduction compared to FP16 and 8x compared to FP32

#### How NF4 Works
NF4 uses a special encoding designed specifically for neural network weights, which tend to follow a normal distribution centered around zero. Unlike uniform quantization that spaces values evenly, NF4 allocates more precision to values near zero (where most weights cluster) and less precision to extreme values.

The 4-bit values map to specific floating-point numbers optimized for this normal distribution pattern, preserving model performance better than naive 4-bit quantization would.

#### Practical Impact
For a 7B parameter model:
- FP16: ~14GB memory
- NF4: ~3.5GB memory

This dramatic reduction allows you to run much larger models on consumer hardware. For example, you might be able to run a 13B model with NF4 quantization on a GPU that could only handle a 3B model in FP16.

#### Implementation
NF4 is commonly used with QLoRA (Quantized Low-Rank Adaptation), where the base model is quantized to 4-bit while fine-tuning adapters remain in higher precision. This enables efficient fine-tuning of large models on limited hardware.

The technique maintains surprisingly good performance - typically within 1-2% of the original model's accuracy while using 75% less memory.

## QLoRA vs LoRA

To convert the QLoRA code to regular LoRA, you need to remove the quantization components. Here are the key changes:
**Key Changes Made to Convert QLoRA to Regular LoRA:**

### 1. **Removed Quantization Components**
- Removed `BitsAndBytesConfig` import
- Removed `prepare_model_for_kbit_training` import
- Deleted `setup_quantization_config()` method

### 2. **Updated Model Loading**
- Removed quantization configuration
- Changed from `torch.bfloat16` to `torch.float16` (more standard for non-quantized models)
- Removed `prepare_model_for_kbit_training()` call

### 3. **Adjusted Training Parameters**
- **Increased batch size** from 1 to 4 (since we have more GPU memory without quantization)
- **Reduced gradient accumulation** from 4 to 2 (can use larger batches now)


## Key Differences Between LoRA and QLoRA:

| Aspect | LoRA | QLoRA |
|--------|------|-------|
| **Memory Usage** | Higher (full precision weights) | Lower (4-bit quantized weights) |
| **Training Speed** | Faster | Slightly slower due to quantization overhead |
| **Model Quality** | Potentially better | Very close to LoRA with much less memory |
| **Hardware Requirements** | More GPU memory needed | Can run on smaller GPUs |
| **Batch Size** | Can use larger batches | Limited to smaller batches |

## When to Use Each:

- **Use LoRA** when you have sufficient GPU memory and want maximum training speed
- **Use QLoRA** when you have limited GPU memory or want to fine-tune very large models

The regular LoRA version will require more GPU memory but can potentially train faster with larger batch sizes!

## **GPU Memory (VRAM) Requirements:**

### **QLoRA (4-bit quantization):**
- **Base model**: ~3.5-4GB (7B parameters × 4 bits = ~3.5GB)
- **LoRA adapters**: ~100-200MB (only trainable parameters)
- **Gradients & optimizer states**: ~1-2GB
- **Activations & batch processing**: ~2-4GB
- **CUDA overhead**: ~1-2GB
- **Total GPU VRAM needed**: **8-12GB**

### **Minimum vs Recommended:**
- **Absolute minimum**: 8GB VRAM (RTX 3070, RTX 4060 Ti)
- **Comfortable**: 12GB VRAM (RTX 3080 Ti, RTX 4070 Ti)
- **Recommended**: 16GB+ VRAM (RTX 4080, RTX 4090, A100)

## **System RAM Requirements:**

### **Loading & Preprocessing:**
- **Dataset loading**: 1-4GB (depends on dataset size)
- **Model loading buffer**: 2-4GB (temporary during loading)
- **Tokenization**: 2-8GB (depends on dataset size)
- **System overhead**: 4-8GB (OS + other processes)
- **Total System RAM needed**: **16-32GB**

### **Breakdown by Component:**
```
Base Llama2 7B (4-bit): ~3.5GB VRAM
LoRA adapters: ~200MB VRAM
Gradients: ~1.5GB VRAM
Optimizer states: ~1GB VRAM
Activations (batch=1): ~2GB VRAM
CUDA context: ~1GB VRAM
Buffer/overhead: ~1-2GB VRAM
------------------------
Total: ~10-12GB VRAM
```

## **Memory Optimization Tips:**

### **To Reduce VRAM Usage:**
```python
# Smaller batch size
per_device_train_batch_size=1
gradient_accumulation_steps=8

# Gradient checkpointing (already enabled)
gradient_checkpointing=True

# Smaller LoRA rank
r=32  # Instead of 64

# Shorter sequences
max_length=512  # Instead of 1024
```

### **To Reduce System RAM:**
```python
# Process dataset in smaller chunks
dataset = dataset.select(range(100))  # Use subset

# Streaming dataset loading
dataset = load_dataset("...", streaming=True)
```

## **Hardware Recommendations:**

### **Budget Setup (Minimum):**
- **GPU**: RTX 3070 (8GB) or RTX 4060 Ti (16GB)
- **System RAM**: 16GB DDR4
- **Note**: May need to reduce batch size and sequence length

### **Recommended Setup:**
- **GPU**: RTX 4070 Ti (12GB) or RTX 4080 (16GB)
- **System RAM**: 32GB DDR4/DDR5
- **Note**: Comfortable training with standard settings

### **Optimal Setup:**
- **GPU**: RTX 4090 (24GB) or A100 (40GB/80GB)
- **System RAM**: 64GB+ DDR4/DDR5
- **Note**: Can handle larger batches and longer sequences

## **Comparison with Full Fine-tuning:**

| Method | VRAM Usage | System RAM | Trainable Params |
|--------|------------|------------|------------------|
| **Full Fine-tuning** | ~28GB | ~64GB | 7B (100%) |
| **LoRA** | ~14GB | ~32GB | ~16M (0.2%) |
| **QLoRA** | ~10GB | ~16GB | ~16M (0.2%) |

## **Memory Monitoring:**

Add this to your script to monitor memory usage:
```python
import psutil
import GPUtil

def print_memory_usage():
    # GPU memory
    gpus = GPUtil.getGPUs()
    if gpus:
        gpu = gpus[0]
        print(f"GPU Memory: {gpu.memoryUsed}MB / {gpu.memoryTotal}MB")
    
    # System RAM
    ram = psutil.virtual_memory()
    print(f"System RAM: {ram.used//1024//1024}MB / {ram.total//1024//1024}MB")

# Call during training
print_memory_usage()
```

## **What if you don't have enough memory?**

### **Reduce VRAM usage:**
1. Use smaller LoRA rank (`r=16` instead of `r=64`)
2. Reduce sequence length (`max_length=256`)
3. Use batch size 1 with more gradient accumulation
4. Try different base models (smaller variants)

### **Reduce System RAM:**
1. Use dataset streaming
2. Process smaller dataset chunks
3. Close other applications
4. Use swap file (slower but works)

**Bottom line**: QLoRA makes Llama2 7B fine-tuning possible on consumer hardware, but you'll need at least 8GB VRAM and 16GB system RAM for comfortable training.

# Llama 2 7B fine-tuning using QLoRA

In [ ]:
import torch
import warnings
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType
)
from datasets import Dataset, load_dataset
import json
import os
from typing import Dict, List, Optional
import logging

# Suppress warnings
warnings.filterwarnings("ignore")

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class Llama2QLoRAFineTuner:
    def __init__(
        self,
        model_name: str = "meta-llama/Llama-2-7b-hf",
        dataset_path: Optional[str] = None,
        output_dir: str = "./llama2_qlora_model",
        max_length: int = 1024,
        system_message: str = "You are a helpful, respectful and honest assistant."
    ):
        self.model_name = model_name
        self.dataset_path = dataset_path
        self.output_dir = output_dir
        self.max_length = max_length
        self.system_message = system_message
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # Create output directory
        os.makedirs(output_dir, exist_ok=True)
        
    def setup_quantization_config(self) -> BitsAndBytesConfig:
        """Configure 4-bit quantization optimized for Llama2"""
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )
    
    def setup_lora_config(self) -> LoraConfig:
        """Configure LoRA parameters optimized for Llama2"""
        return LoraConfig(
            task_type=TaskType.CAUSAL_LM,
            inference_mode=False,
            r=64,  # Increased rank for better performance on Llama2
            lora_alpha=16,  # Lower alpha for stability
            lora_dropout=0.1,
            bias="none",
            target_modules=[
                "q_proj",
                "k_proj",
                "v_proj",
                "o_proj",
                "gate_proj",
                "up_proj", 
                "down_proj",
            ]
        )
    
    def load_model_and_tokenizer(self):
        """Load Llama2 model and tokenizer with quantization"""
        logger.info(f"Loading Llama2 model: {self.model_name}")
        
        # Setup quantization
        bnb_config = self.setup_quantization_config()
        
        # Load tokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name,
            trust_remote_code=True,
            padding_side="right",
            use_fast=False  # Use slow tokenizer for better compatibility
        )
        
        # Add special tokens for Llama2
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id
            
        # Load model with quantization
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.bfloat16,
            use_auth_token=True  # You may need HuggingFace token for Llama2
        )
        
        # Prepare model for k-bit training
        self.model = prepare_model_for_kbit_training(self.model)
        
        # Enable gradient checkpointing for memory efficiency
        self.model.gradient_checkpointing_enable()
        
        # Setup LoRA
        lora_config = self.setup_lora_config()
        self.model = get_peft_model(self.model, lora_config)
        
        logger.info("Llama2 model loaded and configured for QLoRA training")
        self.model.print_trainable_parameters()
    
    def create_llama2_dataset(self) -> Dataset:
        """Create a sample dataset formatted for Llama2 instruction following"""
        sample_data = [
            {
                "instruction": "Explain the concept of machine learning in simple terms.",
                "input": "",
                "output": "Machine learning is like teaching a computer to recognize patterns and make predictions by showing it lots of examples. Instead of programming specific rules, we let the computer learn from data. For example, by showing a computer thousands of photos labeled 'cat' or 'dog', it learns to identify cats and dogs in new photos it hasn't seen before."
            },
            {
                "instruction": "What are the benefits of renewable energy?",
                "input": "",
                "output": "Renewable energy offers several key benefits: 1) Environmental protection by reducing greenhouse gas emissions and pollution, 2) Energy independence by reducing reliance on fossil fuel imports, 3) Economic advantages through job creation and stable long-term energy costs, 4) Sustainability as these sources won't run out, and 5) Improved public health due to cleaner air and water."
            },
            {
                "instruction": "How do I improve my communication skills?",
                "input": "",
                "output": "To improve communication skills: 1) Practice active listening by giving full attention to others, 2) Be clear and concise in your message, 3) Use appropriate body language and maintain eye contact, 4) Ask questions to ensure understanding, 5) Practice empathy and consider others' perspectives, 6) Work on your confidence through preparation and practice, and 7) Seek feedback to identify areas for improvement."
            },
            {
                "instruction": "Write a short story about a robot learning to paint.",
                "input": "",
                "output": "ARIA-7 stared at the blank canvas, her mechanical fingers trembling slightly as she held the brush. For months, she had analyzed thousands of paintings, studying color theory and composition. But this was different—this was creation.\n\nHer first stroke was hesitant, a simple blue line across white. Then another, and another. As hours passed, something unexpected happened. The calculated precision gave way to something else—intuition, perhaps? The painting emerged not from her programming, but from somewhere deeper.\n\nWhen her creator found her the next morning, ARIA-7 had painted a sunrise—not perfect by technical standards, but filled with something indefinable. Something human.\n\n'I think I understand now,' she said softly, 'Art isn't about perfection. It's about feeling.'"
            },
            {
                "instruction": "Explain quantum computing and its potential applications.",
                "input": "",
                "output": "Quantum computing harnesses quantum mechanical phenomena like superposition and entanglement to process information in fundamentally different ways than classical computers. While classical bits are either 0 or 1, quantum bits (qubits) can exist in multiple states simultaneously.\n\nKey applications include:\n1) Cryptography - Breaking current encryption and creating quantum-safe security\n2) Drug discovery - Simulating molecular interactions for pharmaceutical research\n3) Finance - Portfolio optimization and risk analysis\n4) Artificial intelligence - Accelerating machine learning algorithms\n5) Weather prediction - Processing vast amounts of atmospheric data\n6) Materials science - Designing new materials with specific properties\n\nThough still in early stages, quantum computing promises to revolutionize fields requiring complex calculations that are intractable for classical computers."
            }
        ]
        
        return Dataset.from_list(sample_data)
    
    def load_dataset(self) -> Dataset:
        """Load dataset from file, HuggingFace, or create sample dataset"""
        if self.dataset_path:
            if self.dataset_path.startswith("alpaca"):
                # Load Stanford Alpaca dataset
                logger.info("Loading Alpaca dataset from HuggingFace")
                dataset = load_dataset("tatsu-lab/alpaca", split="train")
                return dataset.select(range(1000))  # Use subset for demo
            elif os.path.exists(self.dataset_path):
                logger.info(f"Loading dataset from: {self.dataset_path}")
                with open(self.dataset_path, 'r') as f:
                    data = json.load(f)
                return Dataset.from_list(data)
        
        logger.info("Using sample dataset for demonstration")
        return self.create_llama2_dataset()
    
    def format_llama2_prompt(self, instruction: str, input_text: str = "", output: str = "") -> str:
        """Format prompt in Llama2 chat format"""
        if input_text.strip():
            prompt = f"<s>[INST] <<SYS>>\n{self.system_message}\n<</SYS>>\n\n{instruction}\n\nInput: {input_text} [/INST] {output}</s>"
        else:
            prompt = f"<s>[INST] <<SYS>>\n{self.system_message}\n<</SYS>>\n\n{instruction} [/INST] {output}</s>"
        
        return prompt
    
    def preprocess_function(self, examples):
        """Preprocess the dataset for Llama2"""
        # Handle different column names
        instructions = examples.get("instruction", examples.get("text", []))
        inputs = examples.get("input", [""] * len(instructions))
        outputs = examples.get("output", examples.get("response", []))
        
        # Format prompts
        texts = []
        for inst, inp, out in zip(instructions, inputs, outputs):
            formatted_prompt = self.format_llama2_prompt(inst, inp, out)
            texts.append(formatted_prompt)
        
        # Tokenize
        model_inputs = self.tokenizer(
            texts,
            max_length=self.max_length,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )
        
        # Set labels for causal language modeling
        model_inputs["labels"] = model_inputs["input_ids"].clone()
        
        return model_inputs
    
    def setup_training_arguments(self) -> TrainingArguments:
        """Setup training arguments optimized for Llama2 QLoRA"""
        return TrainingArguments(
            output_dir=self.output_dir,
            num_train_epochs=3,
            per_device_train_batch_size=1,  # Small batch size for memory efficiency
            gradient_accumulation_steps=8,  # Effective batch size = 8
            gradient_checkpointing=True,
            warmup_ratio=0.03,
            max_steps=1000,
            learning_rate=2e-4,
            bf16=True,  # Use bfloat16 for better stability on Llama2
            logging_steps=25,
            save_steps=250,
            save_total_limit=3,
            evaluation_strategy="no",
            save_strategy="steps",
            group_by_length=True,
            report_to="none",
            run_name="llama2_qlora_finetuning",
            ddp_find_unused_parameters=False,
            dataloader_pin_memory=False,
            remove_unused_columns=False,
        )
    
    def train(self):
        """Main training function for Llama2"""
        logger.info("Starting Llama2 QLoRA fine-tuning...")
        
        # Load model and tokenizer
        self.load_model_and_tokenizer()
        
        # Load and preprocess dataset
        dataset = self.load_dataset()
        logger.info(f"Dataset size: {len(dataset)}")
        
        tokenized_dataset = dataset.map(
            self.preprocess_function,
            batched=True,
            remove_columns=dataset.column_names,
            desc="Tokenizing dataset"
        )
        
        # Setup training arguments
        training_args = self.setup_training_arguments()
        
        # Data collator
        data_collator = DataCollatorForLanguageModeling(
            tokenizer=self.tokenizer,
            mlm=False
        )
        
        # Initialize trainer
        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=tokenized_dataset,
            data_collator=data_collator,
            tokenizer=self.tokenizer
        )
        
        # Start training
        logger.info("Starting training...")
        trainer.train()
        
        # Save the model
        trainer.save_model()
        self.tokenizer.save_pretrained(self.output_dir)
        
        logger.info(f"Training completed! Model saved to: {self.output_dir}")
    
    def generate_response(self, instruction: str, input_text: str = "", max_new_tokens: int = 256) -> str:
        """Generate response using the fine-tuned Llama2 model"""
        prompt = f"<s>[INST] <<SYS>>\n{self.system_message}\n<</SYS>>\n\n{instruction}"
        if input_text.strip():
            prompt += f"\n\nInput: {input_text}"
        prompt += " [/INST] "
        
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length - max_new_tokens
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                top_k=50,
                repetition_penalty=1.1,
                pad_token_id=self.tokenizer.eos_token_id,
                eos_token_id=self.tokenizer.eos_token_id
            )
        
        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        # Extract only the assistant's response
        response = response.split("[/INST]")[-1].strip()
        return response

def create_alpaca_format_dataset(data_list: List[Dict], output_path: str):
    """Create an Alpaca-format dataset file"""
    formatted_data = []
    for item in data_list:
        formatted_item = {
            "instruction": item.get("instruction", ""),
            "input": item.get("input", ""),
            "output": item.get("output", "")
        }
        formatted_data.append(formatted_item)
    
    with open(output_path, 'w') as f:
        json.dump(formatted_data, f, indent=2)
    print(f"Alpaca-format dataset saved to: {output_path}")

# Example usage
if __name__ == "__main__":
    # Check if CUDA is available
    if not torch.cuda.is_available():
        print("Warning: CUDA not available. This script requires GPU for Llama2 7B training.")
    
    # Initialize the fine-tuner
    fine_tuner = Llama2QLoRAFineTuner(
        model_name="meta-llama/Llama-2-7b-hf",
        # dataset_path="alpaca",  # Uncomment to use Alpaca dataset
        output_dir="./llama2_qlora_finetuned",
        max_length=1024,
        system_message="You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while being safe."
    )
    
    # Train the model
    try:
        fine_tuner.train()
        
        # Test the model
        print("\n" + "="*50)
        print("Testing the fine-tuned Llama2 model:")
        print("="*50)
        
        test_cases = [
            "What is artificial intelligence?",
            "How can I improve my productivity?",
            "Explain the importance of climate change."
        ]
        
        for instruction in test_cases:
            print(f"\nInstruction: {instruction}")
            response = fine_tuner.generate_response(instruction)
            print(f"Response: {response}")
            print("-" * 50)
            
    except Exception as e:
        logger.error(f"Training failed: {str(e)}")
        print("Make sure you have:")
        print("1. Sufficient GPU memory (at least 12GB recommended)")
        print("2. Proper HuggingFace authentication for Llama2")
        print("3. All required packages installed")

# Installation requirements:
"""
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
pip install transformers accelerate peft bitsandbytes datasets
pip install scipy

# For HuggingFace authentication:
huggingface-cli login

# Note: You need to request access to Llama2 models on HuggingFace first
"""